# 07 · KPIs y gráficos

**Objetivo:** calcular los KPIs a partir de silver, guardarlos en `data/gold/` y contar con ellos la historia del negocio en tres gráficos.

| KPI | Fuente | Tabla gold |
|---|---|---|
| Cuartos noche por mes de llegada y estado | reservaciones | `kpi_cuartos_mes_estado` |
| Cuartos noche por origen | reservaciones | `kpi_cuartos_origen` |
| Ingresos de reservas (completadas) y valor reservado | reservaciones | `kpi_ingresos_reservas` |
| Ingresos de habitaciones, ADR y ADR sin cuartos sin ingreso | ocupaciones | `kpi_produccion_mensual` |

**Reglas que se respetan en todo el notebook**

- Los cocientes (ADR) se calculan **suma entre suma**, nunca promediando.
- Los ingresos de reservaciones son los de las **completadas**: las canceladas conservan su tarifa, pero no es dinero ingresado.
- Los meses `incompleto` (les faltan reservas) y los `futura` (aún podían cancelarse) se marcan en los gráficos.
- `fecha_hoy` en ocupaciones no es la fecha de estancia (S1): se agrupa por ella, pero se llama por su nombre.

## 0 · Preparación

In [ ]:
import sys; sys.path.append("..")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FuncFormatter

from src.rutas import GOLD, SILVER

pd.set_option("display.max_columns", 50)

res = pd.read_parquet(SILVER / "reservaciones.parquet")
oc = pd.read_parquet(SILVER / "ocupaciones.parquet")
print(f"reservaciones: {len(res):,} filas | ocupaciones: {len(oc):,} filas")

In [ ]:
def dividir(numerador, denominador):
    """División que deja vacío (NaN) cuando el denominador es 0, en lugar de dar infinito."""
    return numerador / denominador.replace(0, float("nan"))

## 1 · Calcular los KPIs

### 1.1 · Cuartos noche por mes de llegada y estado

Una fila por mes y estado. Cada mes recibe el `periodo` de su primera llegada, para saber cómo leerlo en el gráfico.

In [ ]:
res["mes_llegada"] = res["fecha_llegada"].dt.to_period("M").dt.to_timestamp()

kpi_mes_estado = (res.groupby(["mes_llegada", "estado_reserva"])["cuartos_noche"].sum()
                     .reset_index())

periodo_del_mes = (res.sort_values("fecha_llegada")
                      .groupby("mes_llegada")["periodo"].first()
                      .reset_index())
kpi_mes_estado = kpi_mes_estado.merge(periodo_del_mes, on="mes_llegada")
kpi_mes_estado.head(10)

### 1.2 · Cuartos noche por origen

Una fila por origen y periodo, con los cuartos noche totales y los de estancias completadas. Para hablar de "de dónde viene la demanda", lo correcto es mirar las **completadas** en el periodo **normal**: es la demanda que de verdad se alojó, en meses con datos completos y sin crisis.

In [ ]:
res["cuartos_completados"] = 0
completada = res["estado_reserva"] == "completada"
res.loc[completada, "cuartos_completados"] = res["cuartos_noche"]

kpi_origen = (res.groupby(["origen_huesped", "periodo"])
                 .agg(cuartos_noche=("cuartos_noche", "sum"),
                      cuartos_completados=("cuartos_completados", "sum"))
                 .reset_index())
kpi_origen.sort_values("cuartos_completados", ascending=False).head(10)

### 1.3 · Ingresos de reservas

Por mes de llegada: lo **ingresado** (tarifa de las completadas) frente al **valor reservado** (tarifa de todas). La diferencia es, sobre todo, el valor de las cancelaciones.

In [ ]:
res["ingresos_completadas"] = 0.0
res.loc[completada, "ingresos_completadas"] = res["tarifa_total"]

kpi_ingresos = (res.groupby("mes_llegada")
                   .agg(ingresos_completadas=("ingresos_completadas", "sum"),
                        valor_reservado=("tarifa_total", "sum"),
                        cuartos_completados=("cuartos_completados", "sum"))
                   .reset_index()
                   .merge(periodo_del_mes, on="mes_llegada"))
kpi_ingresos["tarifa_por_cuarto_noche"] = dividir(kpi_ingresos["ingresos_completadas"], kpi_ingresos["cuartos_completados"])
kpi_ingresos.head()

In [ ]:
# Resumen por periodo: cuánto del valor reservado se convirtió en ingreso
resumen_ingresos = kpi_ingresos.groupby("periodo")[["ingresos_completadas", "valor_reservado"]].sum()
resumen_ingresos["pct_convertido"] = resumen_ingresos["ingresos_completadas"] / resumen_ingresos["valor_reservado"]
resumen_ingresos.reindex(["incompleto", "normal", "covid", "futura"])   # en orden cronológico

### 1.4 · Producción mensual de ocupaciones

Por mes de `fecha_hoy`: ingresos, cuartos noche, cuartos sin ingreso y los dos ADR.

- **ADR** = ingresos / cuartos noche.
- **ADR sin cuartos sin ingreso** = ingresos / (cuartos noche − cuartos sin ingreso). Es el precio medio de lo que sí se cobró.

In [ ]:
oc["mes_fecha_hoy"] = oc["fecha_hoy"].dt.to_period("M").dt.to_timestamp()

kpi_produccion = (oc.groupby("mes_fecha_hoy")
                    .agg(ingresos=("ingresos_habitaciones", "sum"),
                         cuartos_noche=("cuartos_noche", "sum"),
                         cuartos_sin_ingreso=("cuartos_sin_ingreso", "sum"))
                    .reset_index())
kpi_produccion["adr"] = dividir(kpi_produccion["ingresos"], kpi_produccion["cuartos_noche"])
kpi_produccion["adr_sin_cuartos_sin_ingreso"] = dividir(
    kpi_produccion["ingresos"], kpi_produccion["cuartos_noche"] - kpi_produccion["cuartos_sin_ingreso"])
kpi_produccion.head()

## 2 · Validar y guardar gold

Los totales de gold tienen que coincidir con los de silver: si no, alguna agrupación perdió o duplicó filas. Antes de guardar se vacía `data/gold/` para que no queden tablas de versiones anteriores.

In [ ]:
def igual(a, b):
    return abs(a - b) < 0.01

print("Cuartos noche (mes y estado) = silver:", igual(kpi_mes_estado["cuartos_noche"].sum(), res["cuartos_noche"].sum()))
print("Cuartos noche (origen) = silver:      ", igual(kpi_origen["cuartos_noche"].sum(), res["cuartos_noche"].sum()))
print("Ingresos completadas = silver:        ", igual(kpi_ingresos["ingresos_completadas"].sum(), res.loc[completada, "tarifa_total"].sum()))
print("Ingresos de habitaciones = silver:    ", igual(kpi_produccion["ingresos"].sum(), oc["ingresos_habitaciones"].sum()))

In [ ]:
for archivo in GOLD.glob("*.parquet"):
    archivo.unlink()

tablas_gold = {
    "kpi_cuartos_mes_estado": kpi_mes_estado,
    "kpi_cuartos_origen": kpi_origen,
    "kpi_ingresos_reservas": kpi_ingresos,
    "kpi_produccion_mensual": kpi_produccion,
}
for nombre, df in tablas_gold.items():
    df.to_parquet(GOLD / f"{nombre}.parquet", index=False)
    print(f"{nombre}: {len(df):,} filas")

## 3 · Estilo común de los gráficos

Los mismos estados tienen siempre el mismo color en todos los gráficos, y cada título dice la conclusión, no el tema. Las cifras de los títulos se calculan con los datos, así que siempre coinciden con lo que se ve.

In [ ]:
AZUL = "#2a78d6"      # completada / sin cuartos sin ingreso
NARANJA = "#eb6834"   # cancelada / con cuartos sin ingreso
AQUA = "#1baf7a"      # confirmada (futura)
GRIS = "#b4b2a9"      # otros estados
FONDO_CONTEXTO = "#f0efec"
TEXTO = "#0b0b0b"
TEXTO_SUAVE = "#52514e"
REJILLA = "#e6e4df"

def estilo(ax, titulo, subtitulo):
    """Aplica el mismo estilo limpio a todos los gráficos."""
    ax.set_title(titulo, loc="left", fontsize=14, fontweight="bold", color=TEXTO, pad=28)
    ax.text(0, 1.03, subtitulo, transform=ax.transAxes, fontsize=10.5, color=TEXTO_SUAVE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(REJILLA)
    ax.spines["bottom"].set_color(REJILLA)
    ax.tick_params(colors=TEXTO_SUAVE, labelsize=9.5)
    ax.grid(axis="y", color=REJILLA, linewidth=0.8)
    ax.set_axisbelow(True)

def formato_es(valor, decimales=0):
    """Número con punto de miles y coma decimal: 1234567.8 → '1.234.567,8'."""
    texto = f"{valor:,.{decimales}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")

miles = FuncFormatter(lambda valor, _: formato_es(valor))
millones = FuncFormatter(lambda valor, _: formato_es(valor / 1e6, 1) + " M")

## Gráfico 1 · Qué pasó: la demanda se desploma con el COVID

**Pregunta:** ¿cómo evolucionaron los cuartos noche reservados según su mes de llegada?

**Qué muestra:** barras apiladas por estado; las partes suman el total del mes. La zona gris son meses con datos incompletos (S5) y la línea discontinua, la fecha de extracción: a su derecha solo hay cartera que aún podía cancelarse.

In [ ]:
tabla = kpi_mes_estado.pivot_table(index="mes_llegada", columns="estado_reserva",
                                   values="cuartos_noche", aggfunc="sum", fill_value=0)
for estado in ["completada", "cancelada", "confirmada_futura"]:
    if estado not in tabla.columns:
        tabla[estado] = 0
tabla["otros"] = tabla.drop(columns=["completada", "cancelada", "confirmada_futura"]).sum(axis=1)
tabla = tabla.reset_index().merge(periodo_del_mes, on="mes_llegada")

x = list(range(len(tabla)))
etiquetas = tabla["mes_llegada"].dt.strftime("%Y-%m")
idx_incompleto = tabla.index[tabla["periodo"] == "incompleto"]
idx_futura = tabla.index[tabla["periodo"] == "futura"]

abr_jun = tabla[(tabla["mes_llegada"] >= "2020-04-01") & (tabla["mes_llegada"] <= "2020-06-01")]
total_abr_jun = abr_jun[["completada", "cancelada", "confirmada_futura", "otros"]].sum().sum()
pct_cancelado = abr_jun["cancelada"].sum() / total_abr_jun if total_abr_jun else float("nan")

fig, ax = plt.subplots(figsize=(13, 5.5))
if len(idx_incompleto):
    ax.axvspan(idx_incompleto.min() - 0.5, idx_incompleto.max() + 0.5, color=FONDO_CONTEXTO, zorder=0)
    ax.text(idx_incompleto.min() - 0.3, 0.97, "Datos incompletos", transform=ax.get_xaxis_transform(),
            fontsize=9, color=TEXTO_SUAVE, va="top")
if len(idx_futura):
    ax.axvline(idx_futura.min() - 0.5, color=TEXTO_SUAVE, linestyle="--", linewidth=1)
    ax.text(idx_futura.min() - 0.3, 0.97, "Extracción de los datos", transform=ax.get_xaxis_transform(),
            fontsize=9, color=TEXTO_SUAVE, va="top")

base = tabla["completada"]
ax.bar(x, tabla["completada"], width=0.75, color=AZUL, edgecolor="white", linewidth=1, label="Completada")
ax.bar(x, tabla["cancelada"], bottom=base, width=0.75, color=NARANJA, edgecolor="white", linewidth=1, label="Cancelada")
base = base + tabla["cancelada"]
ax.bar(x, tabla["confirmada_futura"], bottom=base, width=0.75, color=AQUA, edgecolor="white", linewidth=1, label="Confirmada (futura)")
base = base + tabla["confirmada_futura"]
ax.bar(x, tabla["otros"], bottom=base, width=0.75, color=GRIS, edgecolor="white", linewidth=1, label="Otros estados")

estilo(ax, f"Entre abril y junio de 2020 se canceló el {pct_cancelado:.0%} de los cuartos noche",
       "Cuartos noche por mes de llegada y estado de la reserva")
ax.set_xticks(x[::2])
ax.set_xticklabels(etiquetas[::2], rotation=45, ha="right")
ax.yaxis.set_major_formatter(miles)
ax.set_ylabel("Cuartos noche", color=TEXTO_SUAVE)
ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=4, frameon=False, fontsize=9.5)
plt.tight_layout()
plt.show()

> **Lectura:** en el periodo normal se completa la gran mayoría de los cuartos noche, con un pico en diciembre–enero. Desde marzo de 2020 crece la parte cancelada y, entre abril y junio, ocupa casi toda la barra. A la derecha de la extracción solo queda cartera.
>
> **Ajusta esta lectura** a lo que veas con tus datos.

## Gráfico 2 · De dónde viene la demanda en un periodo normal

**Pregunta:** ¿qué mercados emisores sostienen el negocio cuando no hay crisis?

**Qué muestra:** el porcentaje de cuartos noche completados por origen, solo en el periodo normal: los 10 primeros y, si hay más, el resto agrupado en "Otros". Los códigos son estados o provincias (AGU = Aguascalientes, WA = Washington…).

In [ ]:
normal = kpi_origen[kpi_origen["periodo"] == "normal"].copy()
normal["pct"] = normal["cuartos_completados"] / normal["cuartos_completados"].sum()
normal = normal.sort_values("pct", ascending=False)

top = normal.head(10)[["origen_huesped", "pct"]]
if len(normal) > 10:
    resto = pd.DataFrame({"origen_huesped": ["Otros"], "pct": [normal["pct"].iloc[10:].sum()]})
    grafico = pd.concat([top, resto], ignore_index=True)
else:
    grafico = top.reset_index(drop=True)

pct_top3 = top["pct"].head(3).sum()
nombres_top3 = ", ".join(top["origen_huesped"].head(3))

fig, ax = plt.subplots(figsize=(10, 6))
posiciones = list(range(len(grafico)))[::-1]     # el primero arriba
colores = [GRIS if nombre == "Otros" else AZUL for nombre in grafico["origen_huesped"]]
ax.barh(posiciones, grafico["pct"], color=colores, height=0.7)
ax.set_yticks(posiciones)
ax.set_yticklabels(grafico["origen_huesped"])
for posicion, valor in zip(posiciones, grafico["pct"]):
    ax.text(valor + 0.003, posicion, f"{formato_es(valor * 100, 1)} %", va="center", fontsize=9.5, color=TEXTO_SUAVE)

estilo(ax, f"{nombres_top3} concentran el {pct_top3:.0%} de las estancias en un periodo normal",
       "% de cuartos noche completados por origen del huésped, periodo normal")
ax.grid(False)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
plt.tight_layout()
plt.show()

> **Tu lectura:** _(escríbela después de ejecutar)_. ¿La demanda se concentra en pocos orígenes o está repartida? ¿Pesa más el mercado nacional o el extranjero? Si todos pesan casi lo mismo, esa es la conclusión honesta, coherente con que los datos parecen generados (S8).

## Gráfico 3 · Ingresos y ADR de ocupaciones

**Pregunta:** ¿cómo evolucionan los ingresos y el precio medio, y cuánto pesan en el ADR los cuartos sin ingreso?

**Qué muestra:** dos paneles con el mismo eje de tiempo. Arriba, los ingresos de habitaciones por mes; abajo, el ADR con y sin los cuartos sin ingreso. Son dos medidas con escalas distintas, así que van en paneles separados en lugar de compartir un eje doble.

**Cuidado al leerlo:** la tabla cubre de noviembre de 2020 a diciembre de 2024, así que es el periodo **posterior** a lo peor del COVID; el mes sale de `fecha_hoy`, que no es la fecha de estancia (S1); y 2024 podría repetir 2021 (pendiente de comprobar).

In [ ]:
prod = kpi_produccion.sort_values("mes_fecha_hoy")
adr_total = prod["ingresos"].sum() / prod["cuartos_noche"].sum()
adr_sin = prod["ingresos"].sum() / (prod["cuartos_noche"] - prod["cuartos_sin_ingreso"]).sum()
efecto = adr_sin / adr_total - 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax1.bar(prod["mes_fecha_hoy"], prod["ingresos"], width=20, color=AZUL)
estilo(ax1, f"Ingresos de habitaciones: {formato_es(prod['ingresos'].sum() / 1e6, 1)} millones en total",
       "Ingresos de habitaciones por mes de fecha_hoy")
ax1.yaxis.set_major_formatter(millones)

ax2.plot(prod["mes_fecha_hoy"], prod["adr_sin_cuartos_sin_ingreso"], color=AZUL, linewidth=2, label="Sin cuartos sin ingreso")
ax2.plot(prod["mes_fecha_hoy"], prod["adr"], color=NARANJA, linewidth=2, label="Con cuartos sin ingreso")
estilo(ax2, f"Sin los cuartos sin ingreso, el ADR sería un {formato_es(efecto * 100, 1)} % más alto",
       "ADR mensual (ingresos / cuartos noche)")
ax2.yaxis.set_major_formatter(miles)
ax2.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2, frameon=False, fontsize=9.5)

plt.tight_layout()
plt.show()

> **Tu lectura:** _(escríbela después de ejecutar)_. ¿Hay estacionalidad en los ingresos? ¿El ADR es estable o tiene tendencia? ¿La distancia entre las dos líneas de ADR es constante?

## 4 · Cierre: hallazgos, limitaciones y recomendaciones

La celda siguiente reúne las cifras clave, calculadas con tus datos, para que el resumen no dependa de copiar números a mano.

In [ ]:
normal_ing = resumen_ingresos.loc["normal"] if "normal" in resumen_ingresos.index else None

print("CIFRAS CLAVE")
print(f"  Cancelado en llegadas de abr–jun 2020:      {pct_cancelado:.0%}")
print(f"  Origen: los 3 primeros ({nombres_top3}):      {pct_top3:.0%} de las estancias (periodo normal)")
if normal_ing is not None:
    print(f"  Valor reservado convertido en ingreso (normal): {normal_ing['pct_convertido']:.0%}")
print(f"  Ingresos de habitaciones (ocupaciones):     {prod['ingresos'].sum():,.2f}")
print(f"  ADR:                                        {adr_total:,.2f}")
print(f"  ADR sin cuartos sin ingreso:                {adr_sin:,.2f} ({efecto:+.1%})")

### Hallazgos

1. **El COVID eliminó la demanda de abril a junio de 2020:** casi todos los cuartos noche con llegada en esos meses se cancelaron (gráfico 1).
2. **Lo reservado no es lo ingresado:** incluso en un periodo normal, solo una parte del valor reservado se convierte en ingreso (tabla de la sección 1.3). La diferencia son, sobre todo, cancelaciones.
3. **Mercados emisores:** _(completa con tu lectura del gráfico 2)_.
4. **Los cuartos sin ingreso rebajan el ADR** en la proporción del gráfico 3. Conviene reportar los dos ADR y explicar la diferencia.

### Limitaciones

- **Ocupación y RevPAR no se pueden calcular:** la capacidad del catálogo de hoteles no cuadra con ningún volumen (I-13).
- **Las dos tablas de hechos no se solapan en el tiempo:** reservaciones cubre estancias hasta julio de 2020 y ocupaciones empieza en noviembre de 2020 (I-21).
- **El significado de los estados de reserva es deducido**, no confirmado (S3).
- **No está confirmado que los cuartos sin ingreso sean cortesías** (I-03).
- **Los datos parecen generados:** las comparaciones entre hoteles o segmentos no reflejarían el negocio (S8).

### Recomendaciones

1. **Negocio:** usar la cancelación del periodo normal como referencia para las políticas de *overbooking* y cancelación, y vigilar la de las llegadas del mes siguiente como indicador temprano de crisis.
2. **Negocio:** reportar como ingreso solo lo realizado (estancias completadas), no el valor reservado.
3. **Datos:** pedir a TCA las capacidades confirmadas de los hoteles y el catálogo de estatus, que desbloquean la ocupación, el RevPAR y la confirmación de los estados.
4. **Datos:** aclarar qué representan `fecha_hoy` y los cuartos sin ingreso.

El registro completo de incidencias y supuestos está en `reports/`, y el proceso, en los notebooks 01 a 06.